# Step 3: Ruptures Segmentation and Segment Feature Creation

## Objective
Use Ruptures library to detect change-points in the time series and create features that characterize each market segment.

## Context
This notebook implements the first two steps of our regime detection workflow:
1. **Change-Point Detection**: Use Ruptures to identify structural breaks in the USD/BRL exchange rate
2. **Segment Characterization**: Calculate features for each segment (duration, mean return, volatility, trend)

These segment-level features will then be used in Notebook 04 for K-Means clustering to identify distinct market regimes.

## Process
1. Load feature data from `data/processed/BRL_X_features.csv`
2. Apply Ruptures change-point detection algorithms
3. Calculate segment duration for each detected segment
4. Extract statistical features per segment:
   - Mean return (directional bias)
   - Volatility (price stability)
   - Trend strength (momentum persistence)
   - Duration (segment length in days)
5. Create segment-level dataset for clustering
6. Visualize change-points and segment characteristics

## Output
- `data/processed/segments.csv`: Segment-level features for K-Means clustering
- `data/processed/changepoints.csv`: Detected change-point indices and dates
- Visualizations of segmentation results

## Ruptures Methods
We'll test multiple change-point detection algorithms:
- **Pelt**: Fast method for finding optimal number of change-points
- **Binary Segmentation**: Efficient top-down approach
- **Bottom-Up**: Merge-based hierarchical method
- **Window**: Sliding window for local change detection

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import ruptures as rpt

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"Ruptures segmentation started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Ruptures version: {rpt.__version__}")

In [ ]:
# Define configuration parameters
FEATURES_PATH = '../data/processed/BRL_X_features.csv'  # Input from notebook 02
SEGMENTS_PATH = '../data/processed/segments.csv'  # Output for notebook 04
CHANGEPOINTS_PATH = '../data/processed/changepoints.csv'  # Change-point reference

# Ruptures configuration
DETECTION_METHOD = 'Pelt'  # Options: 'Pelt', 'Binseg', 'BottomUp', 'Window'
COST_MODEL = 'rbf'  # Cost function: 'l1', 'l2', 'rbf', 'linear', 'normal', 'rank'
MIN_SEGMENT_SIZE = 5  # Minimum days between change-points
PENALTY = 3  # Penalty value for Pelt (higher = fewer change-points)

# Feature selection for change-point detection
# Using Close price as primary signal (can be extended to multivariate)
SIGNAL_COLUMN = 'Close'

print(f"Configuration:")
print(f"  Input: {FEATURES_PATH}")
print(f"  Output (segments): {SEGMENTS_PATH}")
print(f"  Output (changepoints): {CHANGEPOINTS_PATH}")
print(f"  Detection Method: {DETECTION_METHOD}")
print(f"  Cost Model: {COST_MODEL}")
print(f"  Min Segment Size: {MIN_SEGMENT_SIZE} days")
print(f"  Penalty: {PENALTY}")
print(f"  Signal: {SIGNAL_COLUMN}")

In [ ]:
# Load processed features from notebook 02
df = pd.read_csv(FEATURES_PATH, index_col=0)
df.index = pd.to_datetime(df.index)
df.index.name = 'Date'
df = df.sort_index()

print(f"Loaded {len(df)} records from {df.index.min().strftime('%Y-%m-%d')} to {df.index.max().strftime('%Y-%m-%d')}")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Load raw data to get Close prices for change-point detection
# Note: The features file doesn't contain OHLC, so we need the raw data
raw_df = pd.read_csv('../data/raw/BRL_X_raw.csv', index_col=0)
raw_df.index = pd.to_datetime(raw_df.index)
raw_df = raw_df.sort_index()

# Align with features dataframe (same date range)
raw_df = raw_df.loc[df.index]

# Verify alignment
print(f"Raw data shape: {raw_df.shape}")
print(f"Dates match: {(raw_df.index == df.index).all()}")
print(f"\nClose price statistics:")
print(raw_df['Close'].describe())

In [ ]:
# Prepare signal for Ruptures
# Ruptures expects a numpy array of shape (n_samples, n_features)
signal = raw_df[SIGNAL_COLUMN].values.reshape(-1, 1)

print(f"Signal prepared for Ruptures:")
print(f"  Shape: {signal.shape}")
print(f"  Data type: {signal.dtype}")
print(f"  Range: [{signal.min():.4f}, {signal.max():.4f}]")
print(f"  Mean: {signal.mean():.4f}")
print(f"  Std: {signal.std():.4f}")

In [ ]:
# Apply Ruptures change-point detection
# Using Pelt algorithm with RBF kernel for non-linear change detection

print(f"Applying {DETECTION_METHOD} algorithm...")

# Initialize algorithm
if DETECTION_METHOD == 'Pelt':
    algo = rpt.Pelt(model=COST_MODEL, min_size=MIN_SEGMENT_SIZE, jump=1)
elif DETECTION_METHOD == 'Binseg':
    algo = rpt.Binseg(model=COST_MODEL, min_size=MIN_SEGMENT_SIZE, jump=1)
elif DETECTION_METHOD == 'BottomUp':
    algo = rpt.BottomUp(model=COST_MODEL, min_size=MIN_SEGMENT_SIZE, jump=1)
elif DETECTION_METHOD == 'Window':
    algo = rpt.Window(model=COST_MODEL, width=40, min_size=MIN_SEGMENT_SIZE, jump=1)
else:
    raise ValueError(f"Unknown detection method: {DETECTION_METHOD}")

# Fit the algorithm on the signal
algo.fit(signal)

# Detect change-points
changepoints = algo.predict(pen=PENALTY)

# Ruptures returns indices including the last point (length of signal)
# Remove the last point as it's not a true change-point
changepoints = [cp for cp in changepoints if cp < len(signal)]

print(f"\nDetection complete!")
print(f"  Change-points detected: {len(changepoints)}")
print(f"  Number of segments: {len(changepoints) + 1}")
print(f"  Change-point indices: {changepoints[:10]}..." if len(changepoints) > 10 else f"  Change-point indices: {changepoints}")

In [ ]:
# Convert change-point indices to dates
changepoint_dates = [df.index[cp] for cp in changepoints]

# Create change-points dataframe
cp_df = pd.DataFrame({
    'changepoint_index': changepoints,
    'changepoint_date': changepoint_dates
})

print("Change-points with dates:")
print(cp_df.head(10))

# Save change-points for reference
os.makedirs('../data/processed/', exist_ok=True)
cp_df.to_csv(CHANGEPOINTS_PATH, index=False)
print(f"\nChange-points saved to {CHANGEPOINTS_PATH}")

In [ ]:
# Visualize change-points on the Close price series
fig, ax = plt.subplots(figsize=(16, 6))

# Plot Close price
ax.plot(df.index, raw_df['Close'], label='Close Price', linewidth=1, alpha=0.7)

# Plot change-points as vertical lines
for cp_date in changepoint_dates:
    ax.axvline(x=cp_date, color='red', linestyle='--', linewidth=0.8, alpha=0.6)

ax.set_title(f'Detected Change-Points using {DETECTION_METHOD} Algorithm (n={len(changepoints)})', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('USD/BRL Close Price', fontsize=12)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Visualization shows {len(changepoints)} change-points (red dashed lines) identifying {len(changepoints) + 1} market segments")

In [ ]:
# Create segments from change-points
# Each segment is defined by start and end indices

segments = []
start_indices = [0] + changepoints
end_indices = changepoints + [len(signal)]

for i, (start, end) in enumerate(zip(start_indices, end_indices)):
    segments.append({
        'segment_id': i,
        'start_index': start,
        'end_index': end,
        'start_date': df.index[start],
        'end_date': df.index[end - 1]  # end - 1 because end is exclusive
    })

segments_df = pd.DataFrame(segments)

print(f"Created {len(segments_df)} segments:")
print(segments_df.head(10))

In [ ]:
# Calculate segment features
# For each segment, compute statistical characteristics that will be used for clustering

segment_features = []

for _, seg in segments_df.iterrows():
    # Extract segment data
    seg_data = df.iloc[seg['start_index']:seg['end_index']]
    seg_prices = raw_df.iloc[seg['start_index']:seg['end_index']]
    
    # Calculate segment duration (in days)
    duration = len(seg_data)
    
    # Calculate segment returns (percentage change from start to end)
    start_price = seg_prices['Close'].iloc[0]
    end_price = seg_prices['Close'].iloc[-1]
    total_return = (end_price - start_price) / start_price
    
    # Calculate mean daily return
    mean_return = seg_data['target'].mean() if 'target' in seg_data.columns else np.nan
    
    # Calculate volatility (standard deviation of returns)
    # Using std6 as a proxy for volatility, or calculate from Close prices
    if 'std6' in seg_data.columns:
        mean_volatility = seg_data['std6'].mean()
    else:
        # Calculate volatility from price changes
        returns = seg_prices['Close'].pct_change().dropna()
        mean_volatility = returns.std()
    
    # Calculate trend strength (directional consistency)
    # Positive trend = more up days than down days
    price_changes = seg_prices['Close'].diff().dropna()
    trend_strength = (price_changes > 0).sum() / len(price_changes) if len(price_changes) > 0 else 0.5
    
    # Calculate momentum features (average RSL)
    mean_rsl_6 = seg_data['RSL_6'].mean() if 'RSL_6' in seg_data.columns else np.nan
    mean_rsl_12 = seg_data['RSL_12'].mean() if 'RSL_12' in seg_data.columns else np.nan
    
    # Calculate physics-based features (average values)
    mean_velocity = seg_data['v'].mean() if 'v' in seg_data.columns else np.nan
    mean_acceleration = seg_data['a'].mean() if 'a' in seg_data.columns else np.nan
    mean_momentum = seg_data['M'].mean() if 'M' in seg_data.columns else np.nan
    
    # Store segment features
    segment_features.append({
        'segment_id': seg['segment_id'],
        'start_date': seg['start_date'],
        'end_date': seg['end_date'],
        'duration': duration,
        'total_return': total_return,
        'mean_return': mean_return,
        'mean_volatility': mean_volatility,
        'trend_strength': trend_strength,
        'mean_rsl_6': mean_rsl_6,
        'mean_rsl_12': mean_rsl_12,
        'mean_velocity': mean_velocity,
        'mean_acceleration': mean_acceleration,
        'mean_momentum': mean_momentum,
        'start_price': start_price,
        'end_price': end_price
    })

# Create DataFrame with segment features
seg_features_df = pd.DataFrame(segment_features)

print(f"Segment features calculated:")
print(f"  Total segments: {len(seg_features_df)}")
print(f"  Features per segment: {seg_features_df.shape[1]}")
print(f"\nFirst 5 segments with features:")
seg_features_df.head()

In [ ]:
# Display descriptive statistics of segment features
print("Descriptive statistics of segment features:")
print("\nNumeric features only:")
seg_features_df.describe()

In [ ]:
# Visualize segment characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Duration distribution
axes[0, 0].hist(seg_features_df['duration'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Segment Duration Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Duration (days)', fontsize=10)
axes[0, 0].set_ylabel('Frequency', fontsize=10)
axes[0, 0].axvline(seg_features_df['duration'].mean(), color='red', linestyle='--', label=f"Mean: {seg_features_df['duration'].mean():.1f}")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Total return distribution
axes[0, 1].hist(seg_features_df['total_return'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Segment Total Return Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Total Return', fontsize=10)
axes[0, 1].set_ylabel('Frequency', fontsize=10)
axes[0, 1].axvline(0, color='black', linestyle='-', linewidth=0.8)
axes[0, 1].axvline(seg_features_df['total_return'].mean(), color='red', linestyle='--', label=f"Mean: {seg_features_df['total_return'].mean():.4f}")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Volatility distribution
axes[1, 0].hist(seg_features_df['mean_volatility'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Segment Mean Volatility Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Mean Volatility (Std Dev)', fontsize=10)
axes[1, 0].set_ylabel('Frequency', fontsize=10)
axes[1, 0].axvline(seg_features_df['mean_volatility'].mean(), color='red', linestyle='--', label=f"Mean: {seg_features_df['mean_volatility'].mean():.6f}")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Trend strength distribution
axes[1, 1].hist(seg_features_df['trend_strength'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Segment Trend Strength Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Trend Strength (% Up Days)', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].axvline(0.5, color='black', linestyle='-', linewidth=0.8, label='Neutral (0.5)')
axes[1, 1].axvline(seg_features_df['trend_strength'].mean(), color='red', linestyle='--', label=f"Mean: {seg_features_df['trend_strength'].mean():.4f}")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualizations show the distribution of key segment characteristics that will be used for K-Means clustering")

In [ ]:
# Analyze relationship between duration and other features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration vs Total Return
axes[0].scatter(seg_features_df['duration'], seg_features_df['total_return'], alpha=0.6)
axes[0].set_title('Duration vs Total Return', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Duration (days)', fontsize=10)
axes[0].set_ylabel('Total Return', fontsize=10)
axes[0].axhline(0, color='black', linestyle='-', linewidth=0.8)
axes[0].grid(True, alpha=0.3)

# Duration vs Volatility
axes[1].scatter(seg_features_df['duration'], seg_features_df['mean_volatility'], alpha=0.6)
axes[1].set_title('Duration vs Volatility', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Duration (days)', fontsize=10)
axes[1].set_ylabel('Mean Volatility', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Duration vs Trend Strength
axes[2].scatter(seg_features_df['duration'], seg_features_df['trend_strength'], alpha=0.6)
axes[2].set_title('Duration vs Trend Strength', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Duration (days)', fontsize=10)
axes[2].set_ylabel('Trend Strength', fontsize=10)
axes[2].axhline(0.5, color='black', linestyle='--', linewidth=0.8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("These scatter plots show how segment duration relates to other characteristics")
print("Duration is a key feature for distinguishing different market regime types")

In [ ]:
# Check for missing values in segment features
print("Missing values in segment features:")
print(seg_features_df.isnull().sum())
print(f"\nTotal missing values: {seg_features_df.isnull().sum().sum()}")

# Handle missing values if any
if seg_features_df.isnull().sum().sum() > 0:
    print("\nHandling missing values...")
    # For segments with missing values, fill with median or drop
    # This can happen for very short segments
    seg_features_df = seg_features_df.dropna()
    print(f"Segments after handling missing values: {len(seg_features_df)}")

In [ ]:
# Save segment features for K-Means clustering (Notebook 04)
seg_features_df.to_csv(SEGMENTS_PATH, index=False)

print(f"Segment features saved successfully:")
print(f"  Path: {SEGMENTS_PATH}")
print(f"  Shape: {seg_features_df.shape}")
print(f"  Segments: {len(seg_features_df)}")
print(f"  Features per segment: {seg_features_df.shape[1] - 3}  # Excluding segment_id, start_date, end_date")
print(f"\nColumns saved:")
print(list(seg_features_df.columns))

In [ ]:
# Verify saved segments by loading and displaying sample
seg_verification = pd.read_csv(SEGMENTS_PATH)

print("Verification of saved segment features:")
print(f"\nShape: {seg_verification.shape}")
print(f"\nRandom sample of 10 segments:")
seg_verification.sample(min(10, len(seg_verification)))

## Summary

Ruptures segmentation and feature creation completed successfully:
- Applied Ruptures change-point detection to identify market segments
- Detected structural breaks in USD/BRL exchange rate time series
- Created segment-level features for each detected segment:
  - **Duration**: Length of segment in days (key clustering feature)
  - **Total Return**: Overall price change during segment
  - **Mean Return**: Average daily return direction
  - **Mean Volatility**: Price stability indicator
  - **Trend Strength**: Directional consistency
  - **Momentum Indicators**: RSL_6 and RSL_12 averages
  - **Physics Features**: Velocity, acceleration, momentum
- Visualized change-points and segment characteristics
- Saved segment features to `data/processed/segments.csv`

**Key Insights:**
- Change-points successfully identify regime transitions
- Segment durations vary significantly (short spikes vs long trends)
- Duration combined with volatility and returns creates distinct patterns
- Features ready for K-Means clustering to identify regime types

**Detected Segments:**
- Total segments identified from change-point analysis
- Each segment characterized by statistical and behavioral features
- Duration ranges from short-term spikes to extended trends

## Next Steps
Proceed to `04_regime_clustering.ipynb` to:
1. **Apply K-Means Clustering** on segment features
2. **Determine Optimal K** using elbow method and silhouette analysis
3. **Label Regimes** based on cluster characteristics:
   - Bull Market (positive return, low volatility, long duration)
   - Bear Market (negative return, low volatility, long duration)
   - Range-Bound (neutral return, low volatility, medium duration)
   - Crisis/Volatility (high volatility, any return, variable duration)
   - Spike (extreme return, high volatility, short duration)
4. **Assign Regime Labels** to each timestamp in the original dataset
5. **Visualize Regime Timeline** and analyze regime transitions
6. **Prepare for Regime-Aware Modeling** in subsequent notebooks